# Differentially Private Handwritten Digit Recognition (DP-SGD on MNIST)
Trains a CNN on MNIST with **Opacus** DP-SGD (noise multiplier 1.1, clipping norm 1.0) and reports test accuracy per epoch and the final privacy budget ε at δ = 1e-5.

### Setup
Install the pinned dependencies first:
```bash
pip install -r requirements.txt
```

In [1]:
import torch
import torchvision
import opacus
import numpy as np

print("Torch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("Opacus:", opacus.__version__)
print("NumPy:", np.__version__)

Torch: 2.0.1
Torchvision: 0.15.2
Opacus: 1.4.0
NumPy: 1.24.4


In [2]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.optim as optim
from dp_sgd import build_model, make_private

# Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Data loaders
transform = transforms.ToTensor()
train_dataset = datasets.MNIST(root="./data", train=True, transform=transform, download=True)
test_dataset = datasets.MNIST(root="./data", train=False, transform=transform, download=True)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# Model (defined in dp_sgd.py)
model = build_model()
model = model.to(device)

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# Add Differential Privacy (noise multiplier 1.1, max grad norm 1.0)
privacy_engine, model, optimizer, train_loader = make_private(model, optimizer, train_loader)

# Training
def train(model, loader, optimizer, criterion, epoch):
    model.train()
    correct = 0
    for batch_idx, (data, target) in enumerate(loader):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
        pred = output.argmax(dim=1)
        correct += pred.eq(target).sum().item()
    print(f"Epoch {epoch} - Accuracy: {100. * correct / len(loader.dataset):.2f}%")

def test(model, loader):
    model.eval()
    correct = 0
    with torch.no_grad():
        for data, target in loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            pred = output.argmax(dim=1)
            correct += pred.eq(target).sum().item()
    print(f"Test Accuracy: {100. * correct / len(loader.dataset):.2f}%")

# Run training
for epoch in range(1, 6):
    train(model, train_loader, optimizer, criterion, epoch)
    test(model, test_loader)

# Privacy spent
epsilon = privacy_engine.get_epsilon(delta=1e-5)
print(f"\nFinal ε = {epsilon:.2f}, δ = 1e-5")

Using device: cpu
Epoch 1 - Accuracy: 62.81%
Test Accuracy: 80.68%
Epoch 2 - Accuracy: 83.60%
Test Accuracy: 86.29%
Epoch 3 - Accuracy: 86.75%
Test Accuracy: 88.58%
Epoch 4 - Accuracy: 88.35%
Test Accuracy: 88.90%
Epoch 5 - Accuracy: 88.85%
Test Accuracy: 90.02%
Final ε = 0.30, δ = 1e-5
